# Toroidal Alfvén eigenmodes: from field-line tension to Fourier diagnostics

A **toroidal Alfvén eigenmode (TAE)** is a global shear-Alfvén oscillation associated with a gap opened by toroidal geometry in the Alfvén continuum. We will build that statement one step at a time:

1. Derive the shear-Alfvén frequency from magnetic tension.
2. Follow two poloidal harmonics across a tokamak's magnetic surfaces.
3. See why toroidicity couples them and opens a frequency gap.
4. Excite the same harmonics in a small `LinearMHD` simulation.
5. Inspect velocity slices, radial structure, spatial FFTs and time FFTs.
6. Decide what evidence would be needed to identify a TAE.

The analytical figures run in seconds. The simulation uses `(8, 48, 4)` elements and runs to normalized time `20`; allow a few minutes on a laptop with compiled Struphy kernels. Use the Python environment in which Struphy and Jupyter are installed, then **Run All**. This notebook uses the current `Output` Fourier API. No gallery files or downloaded data are needed.

**Scope:** this is a physics and diagnostics tutorial. The short run illustrates an initial-value response; it cannot resolve the estimated TAE period. `LinearMHD` includes the magnetosonic dynamics but no kinetic energetic-particle drive. The prescribed `AdhocTorus` profiles are not a reconstructed tokamak equilibrium. We do not claim a converged ITPA benchmark or a measured TAE eigenfrequency.

In [ ]:
import logging
from tempfile import TemporaryDirectory
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from struphy import (
    BaseUnits, DerhamOptions, EnvironmentOptions, Output, Simulation, Time,
    domains, equils, grids, perturbations, set_logging_level,
)
from struphy.models import LinearMHD
from struphy.post_processing.time_fft import time_fft

set_logging_level(logging.WARNING)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})

## 1. Magnetic tension supplies the restoring force

Imagine displacing a magnetic field line sideways. Bending it produces magnetic tension, while the plasma mass supplies inertia. For a uniform equilibrium, an incompressible transverse displacement obeys

$$
\rho_0\frac{\partial^2\boldsymbol\xi_\perp}{\partial t^2}
=\frac{1}{\mu_0}(\mathbf B_0\cdot\nabla)^2\boldsymbol\xi_\perp.
$$

Insert a plane wave $\boldsymbol\xi_\perp\propto e^{i(k_\parallel s-\omega t)}$:

$$
\omega^2=k_\parallel^2v_A^2,\qquad
v_A=\frac{B_0}{\sqrt{\mu_0\rho_0}}.
$$

The velocity $\mathbf u=\partial_t\boldsymbol\xi$ and magnetic perturbation exchange kinetic and magnetic energy. In a standing wave they are a quarter period out of phase. This is the elementary oscillation from which the TAE is built. The following picture is an analytical standing wave, not simulation output.

In [ ]:
s = np.linspace(0, 2 * np.pi, 300)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
for phase in (0, np.pi / 4, np.pi / 2):
    axes[0].plot(s, np.sin(s) * np.cos(phase), label=fr"$\omega t={phase / np.pi:.2g}\pi$")
axes[0].set(xlabel=r"$k_\parallel s$", ylabel="Transverse displacement / amplitude")
axes[0].legend()
phase = np.linspace(0, 2 * np.pi, 300)
axes[1].plot(phase / (2 * np.pi), np.sin(phase)**2, label="Kinetic")
axes[1].plot(phase / (2 * np.pi), np.cos(phase)**2, label="Magnetic")
axes[1].set(xlabel="Time / wave period", ylabel="Energy / total energy")
axes[1].legend()
plt.show()

## 2. A tokamak has a continuum of local Alfvén frequencies

Let $R_0$ be the major radius, $r$ the minor radius, and $\theta,\phi$ the poloidal and toroidal angles. The safety factor $q(r)$ measures the field-line winding: roughly, a field line makes $q$ toroidal turns per poloidal turn. In a circular, large-aspect-ratio tokamak, choose angle orientations such that a Fourier harmonic is $e^{i(m\theta-n\phi)}$. Then, to leading order,

$$
k_{\parallel,m}(r)\simeq\frac{m/q(r)-n}{R_0},\qquad
\omega_m(r)=\frac{v_A(r)}{R_0}\left|n-\frac{m}{q(r)}\right|.
$$

Different magnetic surfaces have different natural frequencies. These local branches form the **Alfvén continuum**, even before toroidal coupling is included. Neighboring surfaces oscillating at different frequencies also develop finer radial structure over time (phase mixing).

We use the profiles of the small toroidal shear-Alfvén example:

$$
q(r)=1.71+0.16(r/a)^2,\qquad
n_0(r)=1-0.8(r/a)^2,\qquad a=1,\quad R_0=10.
$$

Struphy normalizes magnetic field, density and velocity to reference units; here $B_0=3$ and the single-species mass density has the same normalized profile as $n_0$. Thus the leading-order estimate is $v_A(r)\simeq3/\sqrt{n_0(r)}$, **not** $v_A=1$. The actual toroidal field varies around each surface; we neglect that variation in this uncoupled estimate. Times and angular frequencies below are in normalized Struphy units.

In [ ]:
a, R0, B0 = 1.0, 10.0, 3.0
r_inner, q0, q1 = 0.1, 1.71, 1.87
m, n = 10, 6  # n is the magnitude of the full-torus mode number.

def q_profile(r):
    return q0 + (q1 - q0) * (np.asarray(r) / a)**2

def density_profile(r):
    return 1.0 - 0.8 * (np.asarray(r) / a)**2

def alfven_speed(r):
    return B0 / np.sqrt(density_profile(r))

def continuum(r, poloidal_mode):
    return alfven_speed(r) / R0 * np.abs(n - poloidal_mode / q_profile(r))

q_cross = (m + 0.5) / n
r_cross = a * np.sqrt((q_cross - q0) / (q1 - q0))
omega_tae = alfven_speed(r_cross) / (2 * q_cross * R0)
period_tae = 2 * np.pi / omega_tae
radius = np.linspace(r_inner, a, 600)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
axes[0].plot(radius / a, q_profile(radius), label=r"$q(r)$")
axes[0].axhline(q_cross, color="k", ls=":", label=fr"$q_*={q_cross:.2f}$")
axes[0].set(xlabel=r"$r/a$", ylabel="Safety factor")
axes[0].legend()
for harmonic in (m, m + 1):
    axes[1].plot(radius / a, continuum(radius, harmonic), label=fr"$m={harmonic}$")
axes[1].axvline(r_cross / a, color="k", ls=":")
axes[1].set(xlabel=r"$r/a$", ylabel=r"Local $\omega$", title="Uncoupled continuum estimate")
axes[1].legend()
plt.show()
print(f"Crossing: q = {q_cross:.3f}, r/a = {r_cross/a:.3f}")
print(f"Gap-center estimate: omega = {omega_tae:.5f}, period = {period_tae:.2f}")

## 3. Why do $m=10$ and $m=11$ meet?

At a crossing, the two parallel wavenumbers have equal magnitude and opposite sign:

$$
n-\frac{m}{q_*}=-\left(n-\frac{m+1}{q_*}\right)
\quad\Longrightarrow\quad
q_* = \frac{m+1/2}{n}.
$$

For $m=10$ and $n=6$, this gives $q_*=1.75$ and $r_*/a=0.5$. Each branch has $|k_\parallel|=1/(2q_*R_0)$ there, so the gap-center estimate is

$$
\omega_{\mathrm{TAE}}\simeq\frac{v_A(r_*)}{2q_*R_0}.
$$

This predicts a characteristic frequency, not an eigenvalue of the full radial boundary-value problem. Our density profile gives a period of about **66 time units**. A run ending at $t=20$ covers less than one third of that period.

**Check your intuition:** changing the density rescales the frequencies through $v_A$, but leaves this crossing radius unchanged if $q(r)$ stays fixed.

## 4. Toroidicity couples the harmonics and opens a gap

In a torus, $R=R_0+r\cos\theta$, so the leading toroidal-field variation contains $\cos\theta$. Multiplication by that factor couples adjacent Fourier harmonics because

$$
\cos\theta\,e^{im\theta}=\tfrac12\left(e^{i(m+1)\theta}+e^{i(m-1)\theta}\right).
$$

Coupling removes the crossing, much as two coupled oscillators split into two normal frequencies. This continuum-gap mechanism underlies TAEs; see [Cheng and Chance, *Alfvén continuum with toroidicity* (1986)](https://doi.org/10.1063/1.865926).

To visualize it, we construct an **illustrative two-oscillator matrix**, not the Struphy operator or a quantitative continuum solver:

$$
D(r)=\begin{pmatrix}\omega_m^2&C\\C&\omega_{m+1}^2\end{pmatrix},
\qquad C=\epsilon\,\omega_m\omega_{m+1},\qquad \epsilon=r_*/R_0.
$$

Its eigenvalues are squared frequencies. This choice keeps them nonnegative for $0\le\epsilon<1$. At the crossing, the eigenvectors are equal-weight combinations of the harmonics and $\omega_\pm=\omega_{\mathrm{TAE}}\sqrt{1\pm\epsilon}$. The size of this toy splitting must not be used as a quantitative TAE gap width.

In [ ]:
w_m, w_next = continuum(radius, m), continuum(radius, m + 1)
epsilon = r_cross / R0
D = np.zeros((radius.size, 2, 2))
D[:, 0, 0], D[:, 1, 1] = w_m**2, w_next**2
D[:, 0, 1] = D[:, 1, 0] = epsilon * w_m * w_next
squared_frequency, eigenvectors = np.linalg.eigh(D)
coupled = np.sqrt(np.maximum(squared_frequency, 0))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
for j, ax in enumerate(axes):
    ax.plot(radius / a, w_m, "--", color="0.55", label="Uncoupled")
    ax.plot(radius / a, w_next, "--", color="0.55")
    ax.plot(radius / a, coupled[:, 0], label="Lower coupled branch")
    ax.plot(radius / a, coupled[:, 1], label="Upper coupled branch")
    ax.axvline(r_cross / a, color="k", ls=":")
    ax.set(xlabel=r"$r/a$", ylabel=r"$\omega$", title="Toy avoided crossing" if j == 0 else "Zoom near the crossing")
axes[0].legend(fontsize=8)
axes[1].set(xlim=(0.46, 0.54), ylim=(0.085, 0.107))
plt.show()

## 5. A gap is not yet a global eigenmode

The continuum calculation is local to each magnetic surface. A **global eigenmode** must also have a radial structure that satisfies the radial coupling and boundary conditions. A frequency inside a continuum gap can avoid the local shear-Alfvén resonances that would otherwise transfer energy into fine radial scales. A TAE has coupled poloidal harmonics and a coherent global oscillation. Whether a discrete mode exists, and its frequency and damping, require the global problem.

Energetic particles can drive a TAE when their energy transfer overcomes damping. That is an additional physical mechanism: a TAE need not be unstable, and `LinearMHD` here contains no energetic particles. For a kinetic-MHD example, see [Hou et al., *NIMROD calculations of energetic particle driven toroidal Alfvén eigenmodes* (2017)](https://arxiv.org/abs/1708.05572).

The familiar ITPA geometry uses $R_0/a=10$, $q=1.71+0.16(r/a)^2$ and the $n=6$, $m=10,11$ pair. However, the benchmark has a **flat bulk density** and includes energetic-particle studies. Our inherited `AdhocTorus` density falls radially, and this run only initializes fluid velocity. Sharing the geometry does not reproduce the full benchmark; see the scenario in [Bogaarts et al., *Development and application of a hybrid MHD-kinetic model in JOREK* (2022)](https://pubs.aip.org/aip/pop/article/29/12/122501/2843456/Development-and-application-of-a-hybrid-MHD).

## 6. Map the physics to the Struphy setup

The computational coordinates $\eta_i$ run from 0 to 1. For this hollow torus,

$$
r=0.1+0.9\eta_1,\qquad \theta=2\pi\eta_2.
$$

Only a sixth of the torus is simulated with periodic boundaries. `TorusModes` uses phase $2\pi(m\eta_2+n_{\mathrm{sector}}\eta_3)$. Thus `ns=(-1, -1)` gives full-torus **magnitude** $|n|=6$, not 1. `HollowTorus` uses the Cartesian azimuth $\phi=-2\pi\eta_3/6$; this reverses the toroidal angle relative to the simple convention used in the derivation. The magnitude of the continuum frequency is unaffected when the field and mode conventions are treated consistently.

The initial Gaussian is centered at $\eta_1=0.5$, which means **$r=0.55$**, slightly outside the estimated crossing at $r=0.5$. Its logical width is 0.1 (physical radial width 0.09).

The velocity is initialized in the logical **2-form / H(div) basis**. For each harmonic, its components are proportional to

$$
\widetilde u^1=A\chi(\eta_1)\sin\alpha,\qquad
\widetilde u^2=\frac{A}{2\pi m}\chi'(\eta_1)\cos\alpha,\qquad
\widetilde u^3=0.
$$

Here $\alpha=2\pi(m\eta_2-\eta_3)$ and $\chi$ is the normalized Gaussian used by `TorusModes`. The radial and poloidal derivatives cancel in the logical divergence. This motivates the paired perturbations. It does not prescribe an exact global eigenfunction. These components are also **not** physical unit-vector components; we will push them forward before plotting physical velocity.

In [ ]:
# Increase duration and resolution deliberately after exploring the quick run.
NUM_ELEMENTS = (8, 48, 4)
DEGREE = (3, 3, 2)
DT, END_TIME, SAVE_STEP = 0.5, 20.0, 2

model = LinearMHD(base_units=BaseUnits())
model.propagators.shear_alf.options = model.propagators.shear_alf.Options()
model.propagators.mag_sonic.options = model.propagators.mag_sonic.Options()
for field in (model.em_fields.b_field, model.mhd.density, model.mhd.velocity, model.mhd.pressure):
    field.save_data = True

domain = domains.HollowTorus(
    a1=r_inner, a2=a, R0=R0, sfl=False, pol_period=1, tor_period=6,
)
equil = equils.AdhocTorus(
    a=a, R0=R0, B0=B0, q_kind=0, q0=q0, q1=q1,
    n1=2.0, n2=1.0, na=0.2,  # Make the inherited density defaults explicit.
    p_kind=1, p1=0.95, p2=0.05, beta=0.0018,
)
modes, amplitude = (m, m + 1), 1e-3
model.mhd.velocity.add_perturbation(perturbations.TorusModesSin(
    ms=modes, ns=(-1, -1), amps=(amplitude, amplitude),
    pfuns=("exp", "exp"), pfun_params=([0.5, 0.1], [0.5, 0.1]),
    comp=0, given_in_basis="2",
))
model.mhd.velocity.add_perturbation(perturbations.TorusModesCos(
    ms=modes, ns=(-1, -1), amps=tuple(amplitude / (2 * np.pi * k) for k in modes),
    pfuns=("d_exp", "d_exp"), pfun_params=([0.5, 0.1], [0.5, 0.1]),
    comp=1, given_in_basis="2",
))

## 7. Run the small initial-value problem

The full `LinearMHD` model evolves the perturbation; both the shear-Alfvén and magnetosonic propagators remain active. The spatial and time discretizations are intended for exploration. For example, 48 poloidal elements give only about 4.4 elements per $m=11$ wavelength.

Each execution creates a new temporary output directory. It remains available while the `run_directory` object exists; copy it elsewhere before closing the kernel if you want to keep the data. The runtime limit is 300 minutes to accommodate later, longer experiments; the quick run itself should take a few minutes. Struphy must already have its numerical kernels compiled.

In [ ]:
run_directory = TemporaryDirectory(prefix="struphy_tae_tutorial_")
sim = Simulation(
    model=model,
    name="TAE physics: a short toroidal LinearMHD response",
    description="Coupled m=10,11 velocity perturbations in a periodic one-sixth torus.",
    env=EnvironmentOptions(
        out_folders=run_directory.name, sim_folder="quick_run",
        save_step=SAVE_STEP, max_runtime=300,
    ),
    time_opts=Time(dt=DT, Tend=END_TIME),
    domain=domain, equil=equil,
    grid=grids.TensorProductGrid(num_elements=NUM_ELEMENTS),
    derham_opts=DerhamOptions(
        degree=DEGREE, bcs=(("dirichlet", "dirichlet"), None, None),
    ),
)
started = perf_counter()
output = sim.run(profiling_activated=False)
print(f"Simulation wall time: {perf_counter() - started:.1f} s")
print(f"Output directory: {output.path_out}")

## 8. Recover physical velocity on a poloidal slice

`pproc(physical=True)` creates physical push-forwards. `evaluate` returns a labeled `xarray.DataArray`; it replaces the old post-processing dictionaries. We take the $\phi=0$ slice and rotate Cartesian velocity into orthonormal minor-radial, poloidal and toroidal directions:

$$
u_r=u_x\cos\theta+u_z\sin\theta,\qquad
u_\theta=-u_x\sin\theta+u_z\cos\theta,\qquad u_\phi=u_y.
$$

A denser evaluation grid makes the spline plots smoother but does **not** increase simulation resolution. The two rows below share each component's color scale so that changes in amplitude remain visible.

In [ ]:
output.pproc(physical=True, celldivide=(3, 3, 1), create_vtk=False)
velocity_xyz = output.evaluate("mhd/velocity_xyz").isel(e3=0).transpose("t", "component", "e1", "e2")
assert np.isclose(float(velocity_xyz.t[-1]), END_TIME), "Simulation stopped before END_TIME."
assert np.isfinite(velocity_xyz.values).all(), "Non-finite velocity in the simulation output."

theta = 2 * np.pi * velocity_xyz.e2.values
ux, uy, uz = (velocity_xyz.isel(component=c).values for c in range(3))
components = np.stack((
    ux * np.cos(theta) + uz * np.sin(theta),
    -ux * np.sin(theta) + uz * np.cos(theta),
    uy,
), axis=1)
velocity = velocity_xyz.copy(data=components).assign_coords(component=["r", "theta", "phi"])
velocity = velocity.rename("physical_velocity")
radii = r_inner + (a - r_inner) * velocity.e1.values
velocity = velocity.assign_coords(radius=("e1", radii))
R = R0 + radii[:, None] * np.cos(theta)
Z = radii[:, None] * np.sin(theta)
labels = (r"$u_r$", r"$u_\theta$", r"$u_\phi$")

fig, axes = plt.subplots(2, 3, figsize=(11, 6), constrained_layout=True)
for c, label in enumerate(labels):
    limit = max(float(abs(velocity.isel(component=c)).max()), 1e-16)
    for row, time_index in enumerate((0, -1)):
        field = velocity.isel(t=time_index, component=c)
        artist = axes[row, c].pcolormesh(R, Z, field.values, shading="gouraud", cmap="RdBu_r", vmin=-limit, vmax=limit)
        axes[row, c].set(title=fr"{label}, $t={float(field.t):g}$", xlabel="R", ylabel="Z", aspect="equal")
        axes[row, c].grid(False)
    fig.colorbar(artist, ax=axes[:, c], label="Normalized velocity", shrink=0.75)
plt.show()

## 9. Follow the radial structure

A line at $\theta=\pi/4$ retains the **signed** velocity. A poloidal root-mean-square (RMS) profile instead measures amplitude independent of the angular phase. These answer different questions: use the line to inspect oscillation and the RMS map to track radial localization.

We remove the duplicate periodic endpoint before angular averages and FFTs. These averages are unweighted diagnostics on this slice, not volume-integrated energies. The dashed line marks the estimated continuum crossing; the dotted line marks the initial Gaussian center.

In [ ]:
periodic_indices = np.flatnonzero(velocity.e2.values < 1.0 - 1e-12)
plane = velocity.isel(e2=periodic_indices)
radial_line = plane.sel(e2=0.125, method="nearest")
initial_radius = r_inner + (a - r_inner) * 0.5
selected_times = np.unique(np.linspace(0, plane.sizes["t"] - 1, 4, dtype=int))

fig, axes = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True)
for c, label in enumerate(labels):
    for i in selected_times:
        field = radial_line.isel(t=i, component=c)
        axes[0, c].plot(radii / a, field, label=f"t={float(field.t):g}")
    rms = np.sqrt((plane.isel(component=c)**2).mean("e2"))
    artist = axes[1, c].pcolormesh(radii / a, plane.t.values, rms.values, shading="auto", cmap="magma")
    fig.colorbar(artist, ax=axes[1, c], label=f"RMS {label}")
    axes[0, c].set(title=label, xlabel=r"$r/a$", ylabel="Signed velocity")
    axes[1, c].set(xlabel=r"$r/a$", ylabel="t")
    axes[1, c].grid(False)
    for row in range(2):
        axes[row, c].axvline(r_cross / a, color="tab:green", ls="--", lw=1)
        axes[row, c].axvline(initial_radius / a, color="tab:gray", ls=":", lw=1)
axes[0, 0].legend(fontsize=8)
plt.show()

## 10. Check the seeded poloidal harmonics with a spatial FFT

On the logical interval $\eta_2\in[0,1)$, a Fourier mode $e^{2\pi i m\eta_2}$ has angular wavenumber $k_{\eta_2}=2\pi m$. Therefore divide the returned `k_e2` coordinate by $2\pi$ to obtain the integer poloidal mode number.

Use the **initial logical radial component** for this check. The geometric push-forward and rotation can mix neighboring harmonics in physical components. The real-valued initial field has both positive and negative Fourier modes; we plot the positive half. `fft` returns complex coefficients normalized by the number of samples. For a pure sinusoid with amplitude $A$, each of its two Fourier coefficients has magnitude $A/2$.

In [ ]:
logical_initial = output.evaluate("mhd/velocity").isel(t=0, component=0, e3=0)
logical_initial = logical_initial.isel(e2=periodic_indices)
poloidal_fft = output.fft(logical_initial, dim="e2")
mode_number = poloidal_fft.k_e2.values / (2 * np.pi)
amplitudes = np.sqrt((abs(poloidal_fft)**2).mean("e1")).values
positive_modes = (mode_number >= 0) & (mode_number <= 20)

fig, ax = plt.subplots(figsize=(8, 3), constrained_layout=True)
ax.stem(mode_number[positive_modes], amplitudes[positive_modes], basefmt=" ")
for harmonic in modes:
    ax.axvline(harmonic, color="tab:orange", ls=":")
ax.set(xlabel="Poloidal mode m", ylabel="Radial RMS of |Fourier coefficient|", title="Initial logical radial velocity")
plt.show()

## 11. Time FFT: first check what the record can resolve

For $N$ saved samples separated by $\Delta t_{\rm save}$, the discrete angular-frequency spacing and Nyquist frequency are

$$
\Delta\omega=\frac{2\pi}{N\Delta t_{\rm save}},\qquad
\omega_{\rm Nyquist}=\frac{\pi}{\Delta t_{\rm save}}.
$$

Here $\Delta t_{\rm save}=\mathtt{DT}\times\mathtt{SAVE\_STEP}=1$, not the solver step 0.5. The default record contains the initial state, so $N=21$ and $\Delta\omega\simeq0.299$, while $\omega_{\rm TAE}\simeq0.096$. The target lies below the first nonzero bin. The sample span is $(N-1)\Delta t_{\rm save}$; the DFT's periodic record length is $N\Delta t_{\rm save}$.

The following **synthetic cosine** makes the limitation visible. It uses the estimated frequency but is not a fitted or simulated eigenmode. Increasing the record length resolves the frequency; adding zero padding would only interpolate the existing spectrum. A Hann window reduces endpoint leakage but broadens peaks and changes their power. Struphy does not compensate that window's amplitude or energy loss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), constrained_layout=True)
for count in (21, 1001):
    t_synthetic = np.arange(count, dtype=float)
    signal = xr.DataArray(np.cos(omega_tae * t_synthetic), dims="t", coords={"t": t_synthetic})
    spectrum = time_fft(signal, detrend=True, window="hann")
    normalized_power = spectrum.power / spectrum.power.max()
    axes[0].plot(t_synthetic, signal, "o" if count == 21 else "-",
                 ms=3, zorder=3 if count == 21 else 2, label=f"N={count}")
    axes[1].plot(spectrum.omega, normalized_power, ".-", label=f"N={count}")
    print(f"N={count}: bin spacing = {spectrum.attrs['frequency_resolution']:.5f}")
axes[0].set(xlim=(0, 150), xlabel="t", ylabel="Synthetic signed velocity")
axes[1].axvline(omega_tae, color="k", ls=":", label="Input frequency")
axes[1].set(xlim=(0, 0.65), xlabel=r"$\omega$", ylabel="Power / peak power")
for ax in axes:
    ax.legend(fontsize=8)
plt.show()

## 12. Transform the simulated signed velocity

`output.time_fft(plane)` returns an `xarray.Dataset` with complex `coefficients` and one-sided `power`. Its power is **mean square per frequency bin**, not power spectral density per unit frequency. With no detrending or window, summing it over all frequencies equals the temporal mean square of the original signal (Parseval's identity).

Take the FFT **before** squaring, taking an absolute value or calculating an RMS. For example, $\cos^2(\omega t)=[1+\cos(2\omega t)]/2$: transforming kinetic energy would move the oscillatory peak to $2\omega$.

Below we remove the time mean to display nonzero frequencies, then average the resulting powers over the slice. Averaging signed fields first could cancel the poloidal harmonics. Each component's power is shown in its own units, with no independent peak normalization. The dotted line is a theoretical scale, not a detected peak. Frequency coordinates follow the saved time units; re-evaluate the fields through `output.with_time_units("physical")` to work in seconds and rad/s instead. Passing an existing array always uses that array's time coordinate.

In [ ]:
raw_spectrum = output.time_fft(plane)
np.testing.assert_allclose(
    raw_spectrum.power.sum("omega"), (plane**2).mean("t"), rtol=1e-12, atol=1e-25,
)
# A rectangular time window makes the sparse frequency bins explicit.
spectrum = output.time_fft(plane, detrend=True)
positive = spectrum.power.isel(omega=slice(1, None))
mean_power = positive.mean(("e1", "e2"))
print(f"Saved samples: {plane.sizes['t']}; saved dt: {spectrum.attrs['sample_spacing']:g}")
print(f"Delta omega: {spectrum.attrs['frequency_resolution']:.4f}")
print(f"Nyquist omega: {spectrum.attrs['nyquist_frequency']:.4f}")
print(f"Observed span / estimated TAE period: {float(plane.t[-1] - plane.t[0]) / period_tae:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), constrained_layout=True)
for c, ax in enumerate(axes):
    ax.plot(positive.omega, mean_power.isel(component=c), "o-", ms=4)
    ax.axvline(omega_tae, color="k", ls=":", label="Gap-center estimate")
    ax.set(xlabel=r"$\omega$", ylabel="Mean-square power / bin", title=labels[c], xlim=(0, float(positive.omega[-1])))
axes[0].legend(fontsize=7)
plt.show()

## 13. Where is each frequency located radially?

Average the temporal power over the poloidal angle, retaining radius. A sufficiently long, converged run could show a narrow frequency shared by neighboring radii, with coupled $m=10,11$ structure. Broad features can also arise from the continuum, initial transients and spectral leakage.

For the quick run, interpret this as a demonstration of the diagnostic. Its first positive frequency bin already lies above the predicted gap center. The dashed radial line is $r_*$; it is not a measured eigenmode boundary.

In [ ]:
radial_power = positive.mean("e2")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for c, ax in enumerate(axes):
    values = radial_power.isel(component=c).transpose("omega", "e1")
    artist = ax.pcolormesh(radii / a, positive.omega.values, values.values, shading="auto", cmap="magma")
    ax.axvline(r_cross / a, color="cyan", ls="--", lw=1)
    ax.set(xlabel=r"$r/a$", ylabel=r"$\omega$", title=labels[c])
    ax.grid(False)
    fig.colorbar(artist, ax=ax, label="Mean-square power / bin")
plt.show()

## 14. Reconstruct a selected frequency band

`filter_time` chooses a dominant positive-frequency peak for each component after summing power over the requested dimensions, then retains its contiguous half-power band. The chosen band is shared across the slice. It excludes DC; a component with no oscillatory signal is reported by `has_peak=False`. This is a diagnostic selection, not a TAE classifier.

Compare the filtered signal with the **mean-subtracted** original at a probe near $r_*$ and $\theta=\pi/4$. A short finite record is treated periodically by the FFT, so leakage and edge effects can alter the reconstruction. In particular, the dominant bin of the quick run cannot be interpreted as the TAE frequency.

In [ ]:
band = output.filter_time(plane, dims=("e1", "e2"), pad_bins=0)
radial_index = int(np.argmin(abs(radii - r_cross)))
probe = plane.isel(e1=radial_index).sel(e2=0.125, method="nearest")
filtered_probe = band.filtered.isel(e1=radial_index).sel(e2=0.125, method="nearest")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), constrained_layout=True)
for c, ax in enumerate(axes):
    signal = probe.isel(component=c)
    ax.plot(signal.t, signal - signal.mean("t"), "o-", ms=3, label="Original minus mean")
    ax.plot(filtered_probe.t, filtered_probe.isel(component=c), label="Selected band")
    selected = band.spectrum.isel(component=c)
    if bool(selected.has_peak):
        print(f"{plane.component.values[c]}: retained omega in [{float(selected.omega_lo):.4f}, {float(selected.omega_hi):.4f}]")
    else:
        print(f"{plane.component.values[c]}: no oscillatory peak")
    ax.set(xlabel="t", ylabel="Signed velocity", title=labels[c])
axes[0].legend(fontsize=7)
fig.suptitle(fr"Probe at $r/a={radii[radial_index]/a:.3f}$, $\theta={2*float(probe.e2):.3f}\pi$")
plt.show()

## 15. Turn the illustration into a TAE investigation

Before making an eigenmode claim, improve the independent numerical controls and ask specific physical questions:

1. **Observe several periods.** Try `END_TIME=500` or longer with the small grid first. Compare spectra of later time windows to separate the initial transient from persistent oscillations. Recompute resolution for each shorter analysis window. Resolving a narrow gap can require substantially more time than detecting a single oscillation.
2. **Refine space and time independently.** Compare `(8, 48, 4)` with finer grids; the original larger setup uses `(24, 96, 16)` and degree `(3, 3, 3)`. Reduce `DT` separately. Keep a sufficiently small saved interval to avoid aliasing. More post-processing evaluation points do not replace more simulation elements.
3. **Look for global structure.** Inspect the complex $m=10$ and $m=11$ coefficients versus radius and time. Test whether both share a stable frequency and relative phase. An FFT peak in one probe is insufficient evidence.
4. **Compare with the appropriate continuum.** Replace the illustrative two-oscillator plot with a continuum calculation for the actual equilibrium. Check boundary sensitivity, radial localization and possible continuum resonances. The hollow inner boundary at $r=0.1$ is part of this numerical problem.
5. **Change one physical ingredient.** Use `na=1` for flat density and update `density_profile` as well; predict the frequency change before rerunning. Move the Gaussian toward $r_*$ by setting its logical center to $(r_*-0.1)/0.9$. Changes in excitation should affect mode amplitudes, while a converged eigenfrequency should remain consistent.

The table below only estimates sampling costs; it does not launch longer runs. `END_TIME=500` is a useful next experiment, not a guarantee of convergence or a resolved gap.

In [ ]:
saved_dt = DT * SAVE_STEP
print(f"{'End time':>10} {'Samples':>10} {'TAE periods':>14} {'Delta omega':>14}")
for end_time in (20.0, 500.0, 1000.0):
    count = int(np.floor(end_time / saved_dt)) + 1
    print(f"{end_time:10.0f} {count:10d} {end_time/period_tae:14.2f} {2*np.pi/(count*saved_dt):14.5f}")

## Further reading and next notebooks

- [Cheng and Chance (1986), *Alfvén continuum with toroidicity*](https://doi.org/10.1063/1.865926): the toroidicity-induced continuum gap.
- [Hou et al. (2017), *NIMROD calculations of energetic particle driven toroidal Alfvén eigenmodes*](https://arxiv.org/abs/1708.05572): energetic-particle excitation and the distinction between TAEs and energetic-particle modes.
- [Bogaarts et al. (2022), *Development and application of a hybrid MHD-kinetic model in JOREK*](https://pubs.aip.org/aip/pop/article/29/12/122501/2843456/Development-and-application-of-a-hybrid-MHD): an ITPA benchmark setup and numerical convergence studies.
- [Linear MHD slab waves](tutorial_linear_mhd_slab_waves_1d.ipynb): start with simpler geometry.
- [MHD equilibria](tutorial_mhd_equilibria.ipynb) and [domains](tutorial_domains.ipynb): inspect the equilibrium and coordinate maps used here.
- [Post-processing](tutorial_post_processing.ipynb): explore the current labeled-data workflow.

The saved time histories remain in `output.path_out` until the temporary directory is cleaned up. To preserve them, copy that directory to a permanent location before closing the kernel; reopen it with `Output(path)` for further analysis.